# Verifikasi Data VCT 2021–2026
Notebook ini memeriksa dua klaim di proposal:
1. Struktur file `overview.csv` dan `teams_picked_agents.csv` **sama di keenam tahun**.
2. Tahun **2021–2022 memuat sangat banyak turnamen/komposisi** dan **jumlah agent-nya lebih sedikit** dibanding 2023–2026.

Ubah `BASE` ke folder yang berisi `vct_2021`, `vct_2022`, ..., `vct_2026`.

In [1]:
import pandas as pd
from pathlib import Path

BASE = Path(r".")   # <-- GANTI ke folder yang berisi vct_2021 ... vct_2026
YEARS = [2021, 2022, 2023, 2024, 2025, 2026]

def read(year, sub, name):
    path = BASE / f"vct_{year}" / sub / name
    return pd.read_csv(path, keep_default_na=False, na_values=[''], low_memory=False)

# Cek bahwa semua file ada
for y in YEARS:
    for sub, name in [("matches", "overview.csv"), ("agents", "teams_picked_agents.csv")]:
        p = BASE / f"vct_{y}" / sub / name
        print(y, name, "ADA" if p.exists() else "TIDAK ADA!")

2021 overview.csv ADA
2021 teams_picked_agents.csv ADA
2022 overview.csv ADA
2022 teams_picked_agents.csv ADA
2023 overview.csv ADA
2023 teams_picked_agents.csv ADA
2024 overview.csv ADA
2024 teams_picked_agents.csv ADA
2025 overview.csv ADA
2025 teams_picked_agents.csv ADA
2026 overview.csv ADA
2026 teams_picked_agents.csv ADA


## 1. Apakah struktur kolom sama di keenam tahun?

In [2]:
for sub, name in [("matches", "overview.csv"), ("agents", "teams_picked_agents.csv")]:
    cols = {y: list(read(y, sub, name).columns) for y in YEARS}
    ref = cols[YEARS[0]]
    same = all(c == ref for c in cols.values())
    print(f"{name}: kolom identik di semua tahun? {same}")
    if not same:
        for y, c in cols.items():
            if c != ref:
                print("  beda di", y, "->", c)
    print("  kolom:", ref, "\n")

overview.csv: kolom identik di semua tahun? True
  kolom: ['Tournament', 'Stage', 'Match Type', 'Match Name', 'Map', 'Player', 'Team', 'Agents', 'Rating', 'Average Combat Score', 'Kills', 'Deaths', 'Assists', 'Kills - Deaths (KD)', 'Kill, Assist, Trade, Survive %', 'Average Damage Per Round', 'Headshot %', 'First Kills', 'First Deaths', 'Kills - Deaths (FKD)', 'Side'] 

teams_picked_agents.csv: kolom identik di semua tahun? True
  kolom: ['Tournament', 'Stage', 'Match Type', 'Map', 'Team', 'Agent', 'Total Wins By Map', 'Total Loss By Map', 'Total Maps Played'] 



## 2. Jumlah turnamen, komposisi tim–map, dan agent per tahun
Aturan penyaringan (sama dengan rencana praproses):
- hanya baris `Side == 'both'`
- buang baris ringkasan `Map == 'All Maps'`
- satu komposisi = kombinasi Tournament, Stage, Match Type, Match Name, Map, Team
- komposisi valid = tepat 5 pemain

In [3]:
KEY = ['Tournament', 'Stage', 'Match Type', 'Match Name', 'Map', 'Team']
rows = []
for y in YEARS:
    o = read(y, "matches", "overview.csv")
    b = o[(o['Side'] == 'both') & (o['Map'] != 'All Maps')]

    size = b.groupby(KEY).size()                 # jumlah pemain per komposisi
    valid_keys = size[size == 5].index
    multi = b['Agents'].astype(str).str.contains(',')   # sel berisi >1 agent

    # agent "bersih" = dari sel yang berisi tepat satu agent
    agents_clean = b.loc[~multi, 'Agents'].nunique()

    rows.append({
        'Tahun': y,
        'Turnamen': o['Tournament'].nunique(),
        'Komposisi (semua)': len(size),
        'Komposisi valid (5 pemain)': len(valid_keys),
        'Komposisi tidak valid': int((size != 5).sum()),
        'Sel multi-agent': int(multi.sum()),
        'Agent berbeda': agents_clean,
    })

summary = pd.DataFrame(rows).set_index('Tahun')
summary

,Turnamen,Komposisi (semua),Komposisi valid (5 pemain),Komposisi tidak valid,Sel multi-agent,Agent berbeda
Tahun,,,,,,
2021,142,28768,28620,148,30,16
2022,58,17690,17633,57,27,19
2023,10,1660,1660,0,0,21
2024,15,2208,2208,0,0,24
2025,15,2552,2552,0,0,27
2026,10,1772,1772,0,0,29


## 3. Jumlah agent dari `teams_picked_agents.csv` (pembanding)

In [4]:
ag = {}
for y in YEARS:
    t = read(y, "agents", "teams_picked_agents.csv")
    t = t[t['Map'] != 'All Maps']
    ag[y] = t['Agent'].nunique()
pd.Series(ag, name='Agent berbeda (teams_picked_agents)')

2021    16
2022    19
2023    21
2024    24
2025    27
2026    29
Name: Agent berbeda (teams_picked_agents), dtype: int64

## 4. Kesimpulan otomatis untuk klaim di proposal

In [5]:
early = summary.loc[[2021, 2022]]
recent = summary.loc[[2023, 2024, 2025, 2026]]

print("Turnamen 2021-2022   :", early['Turnamen'].tolist(), "| 2023-2026:", recent['Turnamen'].tolist())
print("Komposisi valid 2021-2022:", early['Komposisi valid (5 pemain)'].tolist(),
      "| 2023-2026:", recent['Komposisi valid (5 pemain)'].tolist())
print("Agent 2021-2022      :", early['Agent berbeda'].tolist(), "| 2023-2026:", recent['Agent berbeda'].tolist())

print()
print("2021-2022 punya lebih banyak turnamen daripada tahun lain? ",
      early['Turnamen'].min() > recent['Turnamen'].max())
print("2021-2022 punya lebih banyak komposisi daripada tahun lain?",
      early['Komposisi valid (5 pemain)'].min() > recent['Komposisi valid (5 pemain)'].max())
print("Agent 2021-2022 lebih sedikit daripada 2023-2026?          ",
      early['Agent berbeda'].max() < recent['Agent berbeda'].min())
print()
print("Total komposisi valid data utama 2023-2026:", int(recent['Komposisi valid (5 pemain)'].sum()))

Turnamen 2021-2022   : [142, 58] | 2023-2026: [10, 15, 15, 10]
Komposisi valid 2021-2022: [28620, 17633] | 2023-2026: [1660, 2208, 2552, 1772]
Agent 2021-2022      : [16, 19] | 2023-2026: [21, 24, 27, 29]

2021-2022 punya lebih banyak turnamen daripada tahun lain?  True
2021-2022 punya lebih banyak komposisi daripada tahun lain? True
Agent 2021-2022 lebih sedikit daripada 2023-2026?           True

Total komposisi valid data utama 2023-2026: 8192


## 5. Contoh nama turnamen (untuk menilai klaim "banyak turnamen regional")
Klaim ini perlu dinilai dengan mata: lihat apakah 2021–2022 memuat banyak turnamen regional/kualifikasi di luar turnamen utama VCT.

In [6]:
for y in [2021, 2022, 2025]:
    o = read(y, "matches", "overview.csv")
    names = o['Tournament'].drop_duplicates().tolist()
    print(f"--- {y}: {len(names)} turnamen, contoh 15 pertama ---")
    for n in names[:15]:
        print("  ", n)
    print()

--- 2021: 142 turnamen, contoh 15 pertama ---
   Valorant Champions 2021
   Champions Tour North America: Last Chance Qualifier
   Champions Tour South America: Last Chance Qualifier
   Champions Tour EMEA: Last Chance Qualifier
   Champions Tour Asia-Pacific: Last Chance Qualifier
   Strike Arabia Championship: Finals
   Valorant Champions Tour Stage 3: Masters Berlin
   Oceania Tour: Championship
   Strike Arabia League: Season 3
   Valorant Conquerors Championship
   Champions Tour SEA Stage 3: Challengers Playoffs
   Champions Tour Brazil Stage 3: Challengers Playoffs
   Champions Tour Stage 3: EMEA Challengers Playoffs
   Champions Tour LATAM Stage 3: Challengers Playoffs
   Champions Tour North America Stage 3: Challengers Playoffs

--- 2022: 58 turnamen, contoh 15 pertama ---
   Valorant Champions 2022
   Champions Tour South America: Last Chance Qualifier
   Champions Tour North America: Last Chance Qualifier
   Champions Tour EMEA: Last Chance Qualifier
   Champions Tour East 